In [ ]:
from datetime import date, datetime, timedelta

from data_engineering.bts.read_bts_ontime_to_flights import get_bts_flights_for_day
from data_engineering.adsbx.get_flights import get_adsbx_flights_for_day
from flights_ai_model.validation import flights_df_comparison
from data_engineering.adsbx.readsb import read_adsbx_readsb
from flights.flights_match_adsb import get_matching_icaos_in_flights
import polars as pl
DAY_MARGIN = timedelta(minutes=30)

test_date = date(2026,3,1)
df_adsb = read_adsbx_readsb()
df_gold = get_bts_flights_for_day(test_date)
df_gold = get_matching_icaos_in_flights(df_gold, df_adsb)
# df_test = get_adsbx_flights_for_day(test_date)
df_test = pl.read_parquet('/Volumes/T2-SSD/planequery/data/flights/trad-model/2026-03-01.parquet')

expr = (pl.col("landing_time").dt.date() == test_date) & (pl.col("takeoff_time").dt.date() == test_date)
min_time = datetime(test_date.year, test_date.month, test_date.day) + DAY_MARGIN
max_time = datetime(test_date.year, test_date.month, test_date.day) + timedelta(days=1) - DAY_MARGIN
expr2 = (pl.col("takeoff_time") > min_time) & (pl.col("landing_time") < max_time)
expr = expr & expr2
df_gold = df_gold.filter(expr)
df_test = df_test.filter(expr)
df_test = df_test.filter(pl.col("icao").is_in(df_gold.get_column("icao")))
df,stats = flights_df_comparison(df_test, df_gold)

6 overlapping flights found for 2026-03-01


In [10]:
COLUMNS = ["icao", "takeoff_time", "takeoff_airport_ident", "landing_time", "landing_airport_ident", "takeoff_time_df2", "takeoff_airport_ident_df2", "landing_time_df2","landing_airport_ident_df2", "match_status"]
df = df.select(COLUMNS)

In [11]:
df.filter(pl.col("match_status") == "df2_only")

icao,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,match_status
str,datetime[ms],str,datetime[ms],str,datetime[ms],str,datetime[ms],str,str
"""a2b8b8""",null,null,null,null,2026-03-01 01:36:00,"""PHNL""",2026-03-01 09:30:00,"""PGUM""","""df2_only"""
"""a42b32""",null,null,null,null,2026-03-01 00:19:00,"""KJAC""",2026-03-01 01:02:00,"""KSLC""","""df2_only"""
"""a6b435""",null,null,null,null,2026-03-01 01:23:00,"""PANC""",2026-03-01 02:40:00,"""PAOM""","""df2_only"""
"""a6ce36""",null,null,null,null,2026-03-01 00:18:00,"""PADK""",2026-03-01 02:40:00,"""PANC""","""df2_only"""
"""a6ed61""",null,null,null,null,2026-03-01 00:19:00,"""KDEN""",2026-03-01 01:50:00,"""KPHX""","""df2_only"""
…,…,…,…,…,…,…,…,…,…
"""a7f63d""",null,null,null,null,2026-03-01 20:13:00,"""PAYA""",2026-03-01 20:52:00,"""PACV""","""df2_only"""
"""aa60da""",null,null,null,null,2026-03-01 00:18:00,"""KSLC""",2026-03-01 00:56:00,"""KCDC""","""df2_only"""
"""ac2fc3""",null,null,null,null,2026-03-01 00:19:00,"""KHDN""",2026-03-01 00:50:00,"""KDEN""","""df2_only"""


In [13]:
df.filter(pl.col("icao") == "a42b32").sort("takeoff_time")

icao,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,match_status
str,datetime[ms],str,datetime[ms],str,datetime[ms],str,datetime[ms],str,str
"""a42b32""",null,null,null,null,2026-03-01 00:19:00,"""KJAC""",2026-03-01 01:02:00,"""KSLC""","""df2_only"""
"""a42b32""",2026-03-01 02:30:42.126,"""KSLC""",2026-03-01 03:28:43.616,"""KDEN""",2026-03-01 02:35:00,"""KSLC""",2026-03-01 03:28:00,"""KDEN""","""matched"""
"""a42b32""",2026-03-01 14:32:41.958,"""KDEN""",2026-03-01 15:47:13.176,"""KSLC""",2026-03-01 14:39:00,"""KDEN""",2026-03-01 15:46:00,"""KSLC""","""matched"""
"""a42b32""",2026-03-01 17:15:05.360,"""KSLC""",2026-03-01 19:32:58.583,"""KAUS""",2026-03-01 17:20:00,"""KSLC""",2026-03-01 19:32:00,"""KAUS""","""matched"""
"""a42b32""",2026-03-01 21:01:58.603,"""KAUS""",2026-03-01 23:28:58.758,"""KMSP""",2026-03-01 21:02:00,"""KAUS""",2026-03-01 23:28:00,"""KMSP""","""matched"""


In [ ]:
df.filter(pl.col("icao") == "a0b59d").sort("takeoff_time")

icao,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,match_status
str,datetime[ms],str,datetime[ms],str,datetime[ms],str,datetime[ms],str,str
"""a0b59d""",2026-03-01 00:50:00,"""KAUS""",2026-03-01 03:38:00,"""KSLC""",2026-03-01 00:49:43.665,"""KAUS""",2026-03-01 03:38:13.849,"""KSLC""","""matched"""
"""a0b59d""",2026-03-01 12:24:00,"""KIDA""",2026-03-01 12:57:00,"""KSLC""",null,null,null,null,"""df1_only"""
"""a0b59d""",2026-03-01 16:13:00,"""KSLC""",2026-03-01 17:48:00,"""KPDX""",2026-03-01 16:08:36.994,"""KSLC""",2026-03-01 17:48:11.812,"""KPDX""","""matched"""
"""a0b59d""",2026-03-01 18:47:00,"""KPDX""",2026-03-01 20:10:00,"""KSLC""",2026-03-01 18:47:43.245,"""KPDX""",2026-03-01 20:10:58.403,"""KSLC""","""matched"""
"""a0b59d""",2026-03-01 21:22:00,"""KSLC""",2026-03-01 23:32:00,"""KAUS""",2026-03-01 21:17:04.017,"""KSLC""",2026-03-01 23:32:58.658,"""KAUS""","""matched"""


In [8]:
df_gold.filter(pl.col("icao") == "a19982")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a19982""",null,"""N202AS""",2026-03-01 01:51:00,"""KPDX""",2026-03-01 07:46:00,"""PHNL""",false,false,false,false,null,null,null,"""A3"""
